In [4]:
import torch
from e2_tts_pytorch import E2TTS, DurationPredictor

from torch.optim import Adam

from e2_tts_pytorch.trainer import (
    TextAudioDataset,
    E2Trainer
)

In [ ]:

# duration_predictor = DurationPredictor(
#     transformer = dict(
#         dim = 80,
#         depth = 2,
#     )
# )

e2tts = E2TTS(
    # duration_predictor = duration_predictor,
    cond_drop_prob = 0.25,
    transformer = dict(
        dim = 512,
        depth = 16,
        heads = 8,
        num_gateloop_layers = 0,
        skip_connect_type = 'concat'
    ),
    mel_spec_kwargs = dict(
        filter_length = 1024,
        hop_length = 256,
        win_length = 1024,
        n_mel_channels = 100,
        sampling_rate = 24000,
    )
)

trainable_param_count = sum(p.numel() for p in e2tts.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_param_count}")

In [ ]:
train_dataset = TextAudioDataset("~/data/LibriTTS_R/dev-clean")

optimizer = Adam(e2tts.parameters(), lr = 1e-4)

trainer = E2Trainer(
    e2tts,
    optimizer,
    num_warmup_steps = 1000,
    checkpoint_path = 'e2tts.pt',
    log_file = 'e2tts.txt'
)

epochs = 50
batch_size = 64
grad_accumulation_steps = 1

print("Training...")

trainer.train(train_dataset, epochs, batch_size, grad_accumulation_steps, num_workers = 0, save_step = 1000)

torch.save(e2tts, "e2tts_test.pt")